# mikltea: offline Double DQN

Run this notebook with **dav3 inside a Slurm H200 allocation** (see `notebook.sbatch`). The game environment is unchanged. The feature extractor is frozen from the scraping teacher.

The three raw features are categorical: mode, suggested direction, useful-bomb flag. We encode them as 10 indicators in **[-1, 1]**, then use a **10 → 32 → 32 → 6** network (1,606 parameters). The network learns an action; the teacher only generated the data and guidance rewards.

Replay loads once per session onto the GPU. Train on solo classic first, then classic against three rule-based opponents. Checkpoints are written atomically after every epoch into this agent's home directory. The final checkpoint is used for gameplay; this notebook does not select a best policy by game score.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

PROJECT = Path('/export/home/ra26xel/bb_rl/bomberman_rl-master')
sys.path.insert(0, str(PROJECT))
from agent_code.mikltea.model import CHECKPOINT, QNetwork
from agent_code.mikltea.offline import Trainer, load_replay

assert torch.cuda.is_available(), 'Select dav3 in a Slurm GPU allocation.'
DEVICE = 'cuda'
print(torch.cuda.get_device_name(), '| parameters:', sum(p.numel() for p in QNetwork().parameters()))
print('Checkpoint:', CHECKPOINT)


## Load a fixed dataset snapshot

The loader reads only shards committed in each manifest, verifies their checksums and feature definitions, and splits **whole episodes** into 90% training / 10% validation. Extensions use unique episode IDs. If scraping is still running, its current committed snapshot is loaded and reported; new shards require reloading this cell. Checkpoints record the exact manifest hashes used.

Smoke datasets and the abandoned 10-million run are deliberately excluded.


In [ ]:
SCRATCH = Path('/export/scratch/ra26xel/bomberman')
STAGE2_PATHS = [SCRATCH / 'triple_scraping_160k/stage_2']
STAGE3_PATHS = [
    SCRATCH / 'triple_scraping_160k/stage_3',
    SCRATCH / 'triple_scraping_stage3_extra200k/stage_3',
    SCRATCH / 'triple_scraping_stage3_extra1640k/stage_3',
]
stage2, sources2 = load_replay(STAGE2_PATHS, DEVICE)
stage3, sources3 = load_replay(STAGE3_PATHS, DEVICE)
print('GPU allocated:', round(torch.cuda.memory_allocated() / 1024**3, 3), 'GiB')


## Reward and training settings

`reward = score + guidance_weight × guidance`. Score includes coins, 5 × opponent kills, and separately recorded posthumous points assigned to the terminal transition. Guidance is the previously proposed scheme: recommended escape/bomb +0.10, recommended coin/target movement +0.05, disagreement −0.05, invalid action −0.10, otherwise zero. There is no separate death penalty.

Start with guidance weight 1.0; it can be lowered later via `trainer.config['guidance_weight']`. Posthumous reward is collapsed onto the last decision rather than discounted by its delay. Both death and the 400-step limit end bootstrapping, matching an episodic game objective.

These are starting hyperparameters, not tuned results. One epoch is approximately one replay-size worth of uniformly sampled transitions **with replacement**. Double DQN uses online action selection and target-network evaluation, Huber loss, Adam, gradient clipping and a hard target update every 250 optimizer steps.


In [ ]:
RESUME = False  # Set True to restore weights, optimizer, RNG, config and epoch history.
STAGE2_EPOCHS = 10
STAGE3_EPOCHS = 20

if RESUME:
    trainer = Trainer.resume(CHECKPOINT, DEVICE)
else:
    if CHECKPOINT.exists():
        raise FileExistsError('A checkpoint exists. Set RESUME=True or archive it before a fresh run.')
    trainer = Trainer(device=DEVICE, lr=3e-4, gamma=.92, guidance_weight=1.0,
                      batch_size=4096, target_interval=250, seed=42)
# Preserve provenance of prior replay snapshots when resuming with more data.
for source in sources2 + sources3:
    if source not in trainer.sources:
        trainer.sources.append(source)


In [ ]:
def plot_losses(history):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot([r['epoch'] for r in history], [r['train_loss'] for r in history], label='Training TD loss')
    ax.plot([r['epoch'] for r in history], [r['validation_loss'] for r in history], label='Held-out TD loss')
    for previous, current in zip(history, history[1:]):
        if previous['stage'] != current['stage']:
            ax.axvline(current['epoch'] - .5, color='gray', linestyle='--')
    ax.set(xlabel='Epoch', ylabel='Huber loss', title=f"{history[-1]['stage']} — {history[-1]['steps']:,} optimizer steps")
    ax.legend()
    display(fig)
    plt.close(fig)
    print(history[-1])
    print('Saved:', CHECKPOINT)

def remaining(stage, target):
    return max(0, target - sum(r['stage'] == stage for r in trainer.history))


In [ ]:
trainer.fit(stage2, remaining('stage_2', STAGE2_EPOCHS), 'stage_2', on_epoch=plot_losses)


In [ ]:
trainer.fit(stage3, remaining('stage_3', STAGE3_EPOCHS), 'stage_3', on_epoch=plot_losses)


## Resume or continue

To resume after an interruption, set `RESUME=True`, reload the replay snapshot, and run the training cells. They skip epochs already completed. Increase `STAGE3_EPOCHS` to continue training. `mikltea.pt` holds online/target weights, optimizer, settings, sampling RNG, loss history and dataset provenance. The plots update after each epoch; interrupting loses at most the current epoch.

**Lower TD loss does not establish stronger gameplay.** Validation measures held-out Bellman error against a moving target. Evaluate points per game against three rule-based opponents on separate seeds before selecting a model. The compressed state is not guaranteed Markov, and offline Q-learning can overvalue poorly sampled state–action pairs.

After training, the engine loads the checkpoint on CPU. Use `--agents mikltea rule_based_agent rule_based_agent rule_based_agent` with your existing Slurm play script. Do not pass `--train`: training here happens offline in this notebook.
